# Advanced RAG: Query Decomposition and Iterative Retrieval

Companion notebook for the [Advanced RAG](https://codemia.ai/courses/applied_ai_systems/advanced_rag) lesson on Codemia.

**What you will build**: a retrieval-augmented generation pipeline that decomposes multi-part questions into sub-queries and iteratively retrieves more context when the first pass is insufficient. Evaluated with simple RAGAS-style metrics.

**Why these two techniques**: naive RAG fails on compound questions ("What year was X founded and who was its first CEO?") because a single embedding smears both concepts together. It also fails on questions where the first retrieval pass misses key context needed to answer a follow-up ("What was the impact of Y?" — you first need to know what Y is). Decomposition and iteration are the smallest meaningful upgrades to the naive baseline.

**References**:
- [RAGAS paper](https://arxiv.org/abs/2309.15217) — automated RAG evaluation metrics
- [LangChain Query Decomposition](https://python.langchain.com/docs/how_to/query_multi_queries/)
- [Self-RAG](https://arxiv.org/abs/2310.11511) — the iterative retrieval idea taken much further

**Time**: ~45 minutes  **GPU required**: no (small embedding model runs on CPU)

## Setup

In [ ]:
%pip install -q sentence-transformers numpy

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import re
import textwrap

np.random.seed(0)

# all-MiniLM-L6-v2 is 80MB, 384-dim embeddings — small enough to run on CPU.
# In production you'd use a larger model (bge-large-en, text-embedding-3-small).
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('embedder loaded, dim =', embedder.get_sentence_embedding_dimension())

## Concept recap

A naive RAG pipeline does: embed query → retrieve top-k → stuff results into a prompt → generate. It fails when:

1. **The query is compound**. "When was Python released and who created it?" is really two queries. Dense retrievers embed the whole thing and land somewhere between the two topics, missing both.
2. **The query needs chained context**. "Why was the GIL added?" requires first finding "what is the GIL" before you can retrieve "why was it added".

**Query decomposition** solves (1) by asking an LLM to split the question into self-contained sub-queries, retrieving for each, then merging.

**Iterative retrieval** solves (2) by checking if the retrieved passages actually answer the question. If not, generate a follow-up query and retrieve again.

## Implementation

We will build up: (1) a tiny corpus, (2) a naive RAG baseline, (3) a query decomposer, (4) an iterative retriever, (5) evaluation metrics.

In [ ]:
# A tiny, handcrafted corpus about programming language history.
# In a real system this would be thousands of Wikipedia chunks; keeping it small
# lets us reason about which passage should match which query.
corpus = [
    "Python was created by Guido van Rossum and first released in 1991. It was designed to emphasize code readability.",
    "The Global Interpreter Lock (GIL) is a mutex in CPython that protects access to Python objects. It prevents multiple native threads from executing Python bytecode simultaneously.",
    "The GIL was originally introduced to make CPython's memory management thread-safe without requiring fine-grained locking on every object.",
    "Rust was created by Graydon Hoare at Mozilla and first released in 2010. Its main goal is memory safety without garbage collection.",
    "The Rust borrow checker enforces ownership rules at compile time, preventing use-after-free and data race bugs.",
    "Go was developed at Google by Robert Griesemer, Rob Pike, and Ken Thompson. It was announced in 2009 and released 1.0 in 2012.",
    "Go has built-in support for concurrency via goroutines and channels, inspired by Tony Hoare's Communicating Sequential Processes (CSP).",
    "JavaScript was created by Brendan Eich at Netscape in 1995. The first version was written in 10 days.",
    "The V8 JavaScript engine was developed by Google for Chrome and uses just-in-time compilation to achieve high performance.",
    "TypeScript was created by Anders Hejlsberg at Microsoft and first released in 2012. It adds optional static typing to JavaScript.",
]
print(f'{len(corpus)} passages in corpus')

In [ ]:
# Pre-compute passage embeddings once. In production this lives in FAISS / pgvector / etc.
corpus_emb = embedder.encode(corpus, normalize_embeddings=True)
print('corpus embeddings:', corpus_emb.shape)

In [ ]:
def retrieve(query, k=3):
    # Dense retrieval with cosine similarity. Because we normalized the embeddings,
    # the dot product IS the cosine similarity.
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    scores = corpus_emb @ q_emb
    top = np.argsort(-scores)[:k]
    return [(corpus[i], float(scores[i])) for i in top]

# Sanity check on a single-topic query.
for p, s in retrieve("Who created the Go programming language?"):
    print(f'{s:.3f}  {p[:80]}')

### Naive RAG baseline (no LLM; template answer)

We will not call an external LLM in this notebook — we want it to run offline. Instead, we simulate the "answer" by returning the top-k passages joined together. The quality metric we will measure is whether the retrieved passages contain the information needed to answer the question — which is exactly what matters for RAG recall.

In [ ]:
def naive_rag(question, k=3):
    retrieved = retrieve(question, k=k)
    passages = [p for p, _ in retrieved]
    return {
        'question': question,
        'passages': passages,
        # In a real pipeline: passages -> LLM prompt -> answer. We skip the LLM and
        # evaluate the retrieval quality directly.
        'answer': ' '.join(passages),
    }

result = naive_rag("When was Python released and who created it?", k=2)
for p in result['passages']:
    print('-', p)

### Query decomposition

We need to detect compound questions and split them. In production this is an LLM call. Here we use a deterministic heuristic so the notebook runs offline — we look for coordinating conjunctions ("and", "as well as") between interrogative clauses. This is crude but captures the key mechanic.

In [ ]:
def decompose_query(question):
    # Heuristic splitter: find interrogative words, split on "and" between them.
    # A real system would use an LLM prompt like:
    #     "Split the following question into self-contained sub-questions: {q}"
    wh_words = r'\b(who|what|when|where|why|how|which)\b'
    parts = re.split(r'\s+and\s+', question, flags=re.IGNORECASE)
    subs = []
    # Keep the subject from the original question so sub-queries are self-contained.
    subject_match = re.search(r'\b(about|of|for|on)\s+([A-Z][\w\s]+?)(?=\?|$|,)', question)
    subject = subject_match.group(2) if subject_match else None
    for p in parts:
        p = p.strip().rstrip('?').strip()
        if not p:
            continue
        # If a part doesn't start with a wh-word, it's a fragment — glue the first part's
        # subject back in.
        if not re.match(wh_words, p, re.IGNORECASE) and subs:
            # Borrow the wh-word from the previous sub.
            p = f'{p}?'
        else:
            p = f'{p}?'
        subs.append(p)
    # If decomposition found only one piece, return the original.
    return subs if len(subs) > 1 else [question]

for q in [
    "When was Python released and who created it?",
    "What is the GIL?",
    "Who created Rust and what is its main goal?",
]:
    print(q)
    for s in decompose_query(q):
        print('  ->', s)

In [ ]:
def decomposed_rag(question, k=2):
    subs = decompose_query(question)
    all_passages = []
    seen = set()
    for sub in subs:
        for p, _ in retrieve(sub, k=k):
            if p not in seen:
                seen.add(p)
                all_passages.append(p)
    return {
        'question': question,
        'sub_queries': subs,
        'passages': all_passages,
        'answer': ' '.join(all_passages),
    }

result = decomposed_rag("When was Python released and who created it?", k=2)
print('sub-queries:', result['sub_queries'])
print()
for p in result['passages']:
    print('-', p)

### Iterative retrieval

After the first retrieval, check whether the passages are sufficient. If not, generate a follow-up query and retrieve again. The "sufficiency check" in production is an LLM critique — here we implement a simple coverage heuristic: do the retrieved passages mention the key nouns from the question?

In [ ]:
def extract_key_terms(text):
    # Pick out capitalized tokens and non-stopword lowercase tokens.
    # A real system would use an LLM or an NER model.
    stop = {'the', 'a', 'an', 'and', 'or', 'of', 'to', 'in', 'is', 'was', 'were', 'be',
            'what', 'when', 'where', 'who', 'why', 'how', 'did', 'does', 'do', 'it',
            'this', 'that', 'for', 'on', 'at', 'by', 'with', 'from', 'as', 'its'}
    words = re.findall(r"[A-Za-z]+", text)
    return {w for w in words if w.lower() not in stop and len(w) > 2}

def coverage(passages, question):
    terms = extract_key_terms(question)
    if not terms:
        return 1.0
    covered = {t for t in terms if any(t.lower() in p.lower() for p in passages)}
    return len(covered) / len(terms)

q = "Why was the GIL introduced in Python?"
passages = [p for p, _ in retrieve(q, k=2)]
print(f'passages: {passages}')
print(f'coverage: {coverage(passages, q):.2f}')

In [ ]:
def iterative_rag(question, k=2, max_rounds=3, coverage_threshold=0.7):
    # Round 1: retrieve on the original question.
    passages = [p for p, _ in retrieve(question, k=k)]
    seen = set(passages)
    rounds = [{'query': question, 'passages': list(passages)}]

    for round_i in range(1, max_rounds):
        cov = coverage(passages, question)
        if cov >= coverage_threshold:
            break
        # Figure out which key terms we still need context for and make a follow-up.
        missing = extract_key_terms(question) - {
            t for t in extract_key_terms(question)
            if any(t.lower() in p.lower() for p in passages)
        }
        if not missing:
            break
        followup = 'What is ' + ' '.join(missing) + '?'
        new = [p for p, _ in retrieve(followup, k=k) if p not in seen]
        for p in new:
            seen.add(p)
            passages.append(p)
        rounds.append({'query': followup, 'passages': new})

    return {
        'question': question,
        'rounds': rounds,
        'passages': passages,
        'answer': ' '.join(passages),
    }

result = iterative_rag("Why was the GIL introduced in Python?")
for i, r in enumerate(result['rounds']):
    print(f'round {i}: {r["query"]}')
    for p in r['passages']:
        print('  -', p[:90])
print(f'final coverage: {coverage(result["passages"], result["question"]):.2f}')

### Evaluation

We will define two RAGAS-style metrics:

- **Context recall**: fraction of the ground-truth answer passages that appear in the retrieved set. This is the most important metric because if the right passage isn't retrieved, the LLM cannot produce a correct answer.
- **Context precision**: fraction of the retrieved passages that are relevant (appear in the ground-truth set). Low precision wastes context window and can confuse the LLM.

The real RAGAS metrics use LLM judges; we use exact-match against hand-labeled gold passages for offline evaluation.

In [ ]:
# Each benchmark item lists which corpus indices are the "gold" passages needed to answer.
benchmark = [
    {'q': "When was Python released and who created it?", 'gold': [0]},
    {'q': "What is the GIL and why was it introduced?", 'gold': [1, 2]},
    {'q': "Who created Rust and what is its main goal?", 'gold': [3]},
    {'q': "How does Go handle concurrency?", 'gold': [6]},
    {'q': "Who created JavaScript and when?", 'gold': [7]},
    {'q': "What is TypeScript and who made it?", 'gold': [9]},
]

def evaluate(pipeline_fn, benchmark, k=3):
    results = []
    for item in benchmark:
        out = pipeline_fn(item['q'], k=k)
        got_passages = set(corpus.index(p) for p in out['passages'] if p in corpus)
        gold = set(item['gold'])
        recall = len(got_passages & gold) / len(gold)
        precision = len(got_passages & gold) / len(got_passages) if got_passages else 0.0
        results.append({'recall': recall, 'precision': precision})
    avg_r = np.mean([r['recall'] for r in results])
    avg_p = np.mean([r['precision'] for r in results])
    return avg_r, avg_p

nr, np_ = evaluate(naive_rag, benchmark, k=2)
dr, dp = evaluate(decomposed_rag, benchmark, k=2)
ir, ip = evaluate(iterative_rag, benchmark, k=2)

print(f'{"pipeline":<20} {"recall":>8} {"precision":>10}')
print(f'{"naive RAG":<20} {nr:>8.2f} {np_:>10.2f}')
print(f'{"decomposed":<20} {dr:>8.2f} {dp:>10.2f}')
print(f'{"iterative":<20} {ir:>8.2f} {ip:>10.2f}')

## Variations

Things to notice and experiment with:

1. **Decomposition helps on compound questions, hurts on simple ones** (it may split where there was nothing to split). An LLM router should check first: "is this a multi-part question?"
2. **Iterative retrieval is slower** — every round is another retrieval + another LLM call in production. Budget: usually 2-3 rounds max.
3. **Coverage threshold is a hyperparameter**. Too high = always iterates, wastes time. Too low = never triggers, same as naive.

In [ ]:
# Try combining decomposition + iteration.
def hybrid_rag(question, k=2):
    subs = decompose_query(question)
    passages = []
    seen = set()
    for sub in subs:
        for p in iterative_rag(sub, k=k)['passages']:
            if p not in seen:
                seen.add(p)
                passages.append(p)
    return {'question': question, 'passages': passages, 'answer': ' '.join(passages)}

hr, hp = evaluate(hybrid_rag, benchmark, k=2)
print(f'hybrid (decompose+iterate): recall={hr:.2f} precision={hp:.2f}')

## What to try next

- Replace the heuristic decomposer with a real LLM call (OpenAI, Anthropic, or a small local model via `transformers`).
- Swap the corpus for a real one: the HotpotQA dev set has multi-hop questions that really exercise iterative retrieval.
- Add a reranker. After dense retrieval, use a cross-encoder (`cross-encoder/ms-marco-MiniLM-L-6-v2`) to rescore the top-20 and keep the top-5. Measure the precision gain.
- Implement [HyDE](https://arxiv.org/abs/2212.10496): generate a hypothetical answer first, embed that, retrieve based on the answer embedding instead of the question embedding.
- Plug in proper RAGAS metrics (`from ragas import evaluate`) with an LLM judge and compare to our rule-based scores.